# Notebook 3 — Entropy as the residue of substrate reading

**Companion to F-004 §13, *Probability as a Time Series*.**

Shannon's entropy is a property of a distribution. If distributions are read off trajectories, then the natural entropy is a property of the *trajectory*: the surprise of the next step conditional on the past, averaged along the way. This notebook makes that distinction operational on two cases that span the picture.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


## 1.  Two definitions

**Shannon entropy** of a distribution $p$:
$$
H_{\mathrm{Shannon}}(X) = -\mathbb{E}[\log p(X)].
$$

**Trajectory entropy** of a process with one-step predictor $\hat M$ and residual $\varepsilon_t = X_{t+1} - \hat M(\text{history}_t)$:
$$
H_{\mathrm{tray}}(\Phi) = -\mathbb{E}_t[\log p(X_{t+1} \mid \text{history}_t)] = -\mathbb{E}_t[\log p_\varepsilon(\varepsilon_t)].
$$

The two coincide when the substrate has nothing to read. They differ by exactly the operational information the substrate extracts from the past.

## 2.  Fair coin: substrate reads nothing, Shannon is the right measure


In [ ]:
rng = np.random.default_rng(42)
N = 20000
X = rng.integers(0, 2, size=N).astype(float)  # 0/1 i.i.d.

# Shannon entropy of the marginal
p = np.array([(X == 0).mean(), (X == 1).mean()])
H_shannon = -np.sum(p * np.log(p))

# Trajectory entropy: conditional given previous toss
H_tray = 0.0
for prev in [0.0, 1.0]:
    mask = (X[:-1] == prev)
    if mask.sum() == 0: continue
    future = X[1:][mask]
    p1 = future.mean()
    pc = np.array([1-p1, p1])
    H_tray += -np.sum(pc * np.log(pc + 1e-12)) * mask.mean()

print(f'fair coin, N = {N}')
print(f'  H_Shannon = {H_shannon:.4f}   (= log 2 = {math.log(2):.4f})')
print(f'  H_tray    = {H_tray:.4f}')
print(f'  gap       = {H_shannon - H_tray:.4f}   (theory: 0)')
print()
print('The substrate has nothing to read; the two entropies agree.')


## 3.  AR(1): substrate reads the dynamics, entropy collapses to the innovation

For $X_{t+1} = \phi X_t + \sigma z_t$ with $z_t \sim \mathcal{N}(0,1)$:
- Marginal $X$ is $\mathcal{N}(0, \sigma^2/(1-\phi^2))$, with $H_{\mathrm{Shannon}} = \tfrac12 \log(2\pi e\, \sigma^2/(1-\phi^2))$.
- Innovation $\varepsilon = \sigma z_t$ has $H_{\mathrm{tray}} = \tfrac12 \log(2\pi e\, \sigma^2)$.
- Gap: $-\tfrac12 \log(1 - \phi^2)$.


In [ ]:
def ar1_entropies(phi, n=20000, sigma=1.0, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n) * sigma
    X = np.zeros(n)
    for t in range(1, n):
        X[t] = phi * X[t-1] + z[t]
    Hs = 0.5 * math.log(2*math.pi*math.e * X.var())
    A = X[:-1].reshape(-1, 1)
    b = X[1:]
    phi_hat, *_ = np.linalg.lstsq(A, b, rcond=None)
    eps = b - A @ phi_hat
    Ht = 0.5 * math.log(2*math.pi*math.e * eps.var())
    return Hs, Ht, phi_hat[0]

print(f'{"phi":<5s}  {"phi_hat":<8s}  {"H_Shannon":<11s}  {"H_tray":<9s}  {"gap":<8s}  {"theory":<8s}')
print('-' * 60)
for phi in [0.0, 0.3, 0.6, 0.9, 0.99]:
    Hs, Ht, phi_h = ar1_entropies(phi)
    gap_th = -0.5 * math.log(1 - phi**2) if phi**2 < 1 else float('inf')
    print(f'{phi:<5.2f}  {phi_h:<8.4f}  {Hs:<11.4f}  {Ht:<9.4f}  {Hs-Ht:<8.4f}  {gap_th:<8.4f}')


The substrate recovers $\phi$ from the trajectory; the trajectory entropy stays at the innovation level $\tfrac12 \log(2\pi e)$, while the marginal Shannon entropy inflates as $\phi \to 1$.

## 4.  The gap visualised


In [ ]:
phis = np.linspace(0.0, 0.99, 50)
Hs = []; Ht = []
for phi in phis:
    s, t, _ = ar1_entropies(float(phi))
    Hs.append(s); Ht.append(t)
Hs = np.array(Hs); Ht = np.array(Ht)
gap_theory = -0.5 * np.log(1 - phis**2)
base = 0.5 * math.log(2 * math.pi * math.e)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(phis, Hs, lw=2.2, color='#222', label=r'$H_{\mathrm{Shannon}}$ (marginal)')
ax.plot(phis, Ht, lw=2.2, color='#3a7ca5', label=r'$H_{\mathrm{tray}}$ (residual)')
ax.plot(phis, base + gap_theory, lw=1.0, ls='--', color='#a83232',
        label=r'theory: $\frac{1}{2}\log(2\pi e) - \frac{1}{2}\log(1-\phi^2)$')
ax.fill_between(phis, Ht, Hs, color='#3a7ca5', alpha=0.10,
                label='gap = information $\hat M$ reads')
ax.set_xlabel(r'$\phi$', fontsize=11); ax.set_ylabel('entropy (nats)', fontsize=11)
ax.set_title('Trajectory entropy vs Shannon entropy on AR(1)', fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(1.3, 4.0); ax.grid(alpha=0.3)
ax.legend(loc='upper left', fontsize=9, framealpha=0.95)
plt.tight_layout(); plt.show()


The shaded gap is $-\tfrac12 \log(1-\phi^2)$ exactly, the information the operator reads from the past.

## 5.  Relation to other entropies

The trajectory entropy fits into a family of trajectory-flavoured information measures:

- **Conditional Shannon entropy** $H(X_{t+1} \mid X_t)$ — same idea, with the conditional distribution estimated nonparametrically. $H_{\mathrm{tray}}$ specialises to "the predictor is the substrate's OLS fit on the kinematic jet" instead of a generic conditional distribution.
- **Entropy rate** $\lim_n \tfrac{1}{n} H(X_1, \ldots, X_n)$ — for stationary processes this equals $H_{\mathrm{tray}}$ in the limit. The substrate provides a *constructive* estimator: fit the operator, read the residual.
- **Kolmogorov–Sinai entropy** of a dynamical system — entropy of partitions of state space along the trajectory. $H_{\mathrm{tray}}$ is the OLS-residue version, applied to the kinematic-jet partition.
- **KL divergence** $D_{KL}(p \| q)$ between two distributions — the trajectory reading is "by how much does the substrate of $p$ misfit $q$'s trajectory". A natural follow-up but not pursued here.

What is specific to $H_{\mathrm{tray}}$ is the substrate as observer. Shannon's measure is the entropy of trajectories that no substrate has read. The trajectory entropy is what remains once the substrate has done its work. The quantity is observer-relative, and the observer, in this construction, is the algebra.

## 6.  Take-away

- The fair coin: $H_{\mathrm{tray}} = H_{\mathrm{Shannon}} = \log 2$. The substrate gives up; Shannon is right.
- The AR(1): $H_{\mathrm{tray}} \ll H_{\mathrm{Shannon}}$, with closed-form gap $-\tfrac12 \log(1-\phi^2)$. The substrate reads the dynamics; what is left is the innovation noise.
- The reformulation does not replace Shannon — it relocates entropy from the marginal to the residual of an algebraic predictor.
